<a href="https://colab.research.google.com/github/catastrxphic/AI_Practice/blob/ImageGenerator_StableDiffusion/ImageGenerator_stableDiffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Playing Around With Stable Diffusion's Code

### Aim:
###### Get a better understanding of what happens behind the curtains when creating images from text with AI

## Initializing Stable Diffusion

In [ ]:
''' installations needed:
- lpips (learned perceptual image patch similarity) -> judge simmilarity between images
- accelerate -> speed up training & improve GPU use
- diffusers -> generate images from text
'''
!pip install transformers diffusers lpips accelerate

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Imports & use:

# DS for tensors and deep learning model development functionalities
import torch
# encode text into embeddings and coverst text to numbers the model understands
from transformers import CLIPTextModel, CLIPTokenizer
# encode and decode images, denoising network in diffusion process, manage noise scheduling on diffusion steps
from diffusers import AutoencoderKL, UNet2DConditionModel, LMSDiscreteScheduler
# display progress bars
from tqdm.auto import tqdm
# enables faster computations with lower memory use
from torch import autocast
from matplotlib import pyplot as plt
# image manipulation
from PIL import Image
import numpy as np
# image transformations fro preprocessing
from torchvision import transforms as tfms

# video display
from IPython.display import HTML
from base64 import b64encode

# setting device:
torch_device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# load autoencoder to help decode latents into image space
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae", use_auth_token=True)

# load tokenizer & text encoder to encode and tokenize text
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14")

# load UNET model to generate latents
unet = UNet2DConditionModel.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="unet", use_auth_token=True)

# load noise scheduler
scheduler = LMSDiscreteScheduler(beta_start=0.00085, beta_end=0.012, beta_schedule="scaled_linear", num_train_timesteps=1000)

# go to GPU
vae = vae.to(torch_device)
text_encoder = text_encoder.to(torch_device)
unet = unet.to(torch_device)

## Inference

In [ ]:
# import drive module
from google.colab import drive
# bridging google drive and colab env
drive.mount('/content/drive')

In [ ]:
# Variables for image generation
prompt = [""]
height = 512
width = 768
inference_steps = 50
guidance_scale = 7.5
generator = torch.manual_seed(4)
batch_size = 1

In [ ]:
# prepate text
text_input = tokenizer(prompt, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")
# torch no grad disables gradient calculation temporarily
with torch.no_grad():
  # encode and save text as encoded numbers
  text_embedding = text_encoder(text_input.input_ids.to(torch_device))[0]
max_len = text_input.input_ids.shape[-1]
uncond_input = tokenizer(
    [""] * batch_size, padding="max_length", max_length=max_len, return_tensors="pt"
)

with torch.no_grad():
  uncond_embeddings = text_encoder(uncond_input.input_ids.to(torch_device))[0]
text_embedding = torch.cat([uncond_embeddings, text_embedding])

In [ ]:
# # preparing scheduler
# scheduler.set_timesteps(inference_steps)

# # preparing latents
# latents = torch.randn(
#     (batch_size, unet.in_channels, height // 8, width // 8),
#     generator=generator,
# )

In [ ]:
# latents = latents.to(torch_device)
# latents = latents * scheduler.sigmas[0] # Need to scale to match k

#     # Loop
# with autocast("cuda"):
#     for i, t in tqdm(enumerate(scheduler.timesteps)):
#         # expand the latents if we are doing classifier-free guidance to avoid doing two forward passes.
#         latent_model_input = torch.cat([latents] * 2)
#         sigma = scheduler.sigmas[i]
#         latent_model_input = latent_model_input / ((sigma**2 + 1) ** 0.5)

#         # predict the noise residual
#         with torch.no_grad():
#             noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embedding)["sample"]

#         # perform guidance
#         noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
#         noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

#         # compute the previous noisy sample x_t -> x_t-1
#         latents = scheduler.step(noise_pred, i, latents)["prev_sample"]

# # scale and decode the image latents with vae
# latents = 1 / 0.18215 * latents

# with torch.no_grad():
#     image = vae.decode(latents)

# # Display
# image = (image / 2 + 0.5).clamp(0, 1)
# image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
# images = (image * 255).round().astype("uint8")
# pil_images = [Image.fromarray(image) for image in images]
# pil_images[0]

In [ ]:
# preparing scheduler
scheduler.set_timesteps(inference_steps)

# preparing latents
latents = torch.randn(
    (batch_size, unet.in_channels, height // 8, width // 8),
    generator=generator,
)

latents = latents.to(torch_device)
latents = latents * scheduler.init_noise_sigma # changed sigmas[0] to init_noise_sigma

# Loop
with autocast("cuda"):
    for i, t in tqdm(enumerate(scheduler.timesteps)):
        # expand the latents if we are doing classifier-free guidance to avoid doing two forward passes.
        latent_model_input = torch.cat([latents] * 2)
        latent_model_input = scheduler.scale_model_input(latent_model_input, t) #moved out of loop

        # predict the noise residual
        with torch.no_grad():
            noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embedding)["sample"]

        # perform guidance
        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

        # compute the previous noisy sample x_t -> x_t-1
        latents = scheduler.step(noise_pred, t, latents)["prev_sample"] # changed i to t

In [ ]:
# scale and decode the image latents with vae
latents = 1 / 0.18215 * latents

with torch.no_grad():
    image = vae.decode(latents)

# Display
image = (image / 2 + 0.5).clamp(0, 1)
image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
images = (image * 255).round().astype("uint8")
pil_images = [Image.fromarray(image) for image in images]
pil_images[0]